# Lab 8: Machine Translation

## 1 Getting Started

In today’s lab you will be looking at ways of evaluating
machine translation.

There are two plain text files provided. One `HarryPotter-en` provides the first ten sentences of the first Harry Potter novel (one sentence per line). The second `HarryPotter-fr` provides the same ten sentences translated into French by a human - i.e., this is a gold standard translation.

## 2 Tasks

#### 1. Use Google Translate (or some other translation system) to automatically translate the English into French. Copy the output into a plain text file where there is one sentence per line. Make a second file which is an automatic translation of the French into English.

- `HarryPotter-en.txt`
- `HarryPotter-ft.txt`
- `HarryPotter-en_to_fr.txt`
- `HarryPotter-fr_to_en.txt`

---

#### 2. Write a function to compute the unigram precision for a sentence i.e., the proportion of unigrams in the automatic translation which are also found in the gold standard translation. Use your function to compute the unigram precision for each sentence in a file and return the average over the whole file. Test out your code on both the English-French translation and the French-English translation and comment on the difference.

To complete Task 2, we need to build a metric that measures word-for-word overlap. We are simply checking that words in the translation exist in the gold-standard "label". 

This is a Precision based metric, recalling the Precision is the ratio of the True Positives against all of the positive predictions that the model made, i.e. True Postives and False Positives. By definition we have to go down the route of Precision because a translatation representation a models predictions. 

Conversely, whilst we have gold-standard label from which we could conduct Recall, it is often unreasonable from a language evaluation perspective because there are many ways to phrase a sentence and gold-standard is just one of them. A translation could be completely valid but use synonymns to the label. However, if we use the label as a guide for Precision, whilst it only represents 1 possible translation, all other translations are compared to and normailsed against the label, which if it is a good translation should hold some valuable words. 

**Numerator:** The number of words in the automatic translation that also appear in the gold-standard. (TP)

**Denominator:** The total number of words in the automatic translation. (TP + FP)

#### Implementation
1. **Tokenization:** Probably just stick with case normalised tokenization. More advanced methods could look at subword tokenization or morphological tokenization.
2. **Overlap Calculation:** Use a loop or set intersection to find common words.
3. **File Iteration:** Read both the machine-translated file and the gold standard file line-by-line, calculate precision for each pair, and then compute the mean.





In [1]:
def compute_unigram_precision(candidate, reference):
    """
    Computes precision for a single sentence.
    candidate: string (machine translation)
    reference: string (gold standard)
    """
    # Simple tokenization (lowercasing and splitting by whitespace)
    cand_words = candidate.lower().split()
    ref_words = reference.lower().split()
    
    if len(cand_words) == 0:
        return 0
    
    # Count how many words in candidate exist in reference
    # Note: Task 2 uses simple precision, so we don't worry about repeating words yet
    matches = [word for word in cand_words if word in ref_words]
    
    return len(matches) / len(cand_words)

def evaluate_file_precision(candidate_file, reference_file):
    """
    Computes the average unigram precision over entire files.
    """
    with open(candidate_file, 'r', encoding='utf-8') as cand_f, \
         open(reference_file, 'r', encoding='utf-8') as ref_f:
        
        cand_lines = cand_f.readlines()
        ref_lines = ref_f.readlines()
        
        total_precision = 0
        count = 0
        
        for cand, ref in zip(cand_lines, ref_lines):
            total_precision += compute_unigram_precision(cand.strip(), ref.strip())
            count += 1
            
        return total_precision / count if count > 0 else 0

# Test the function on your files
en_to_fr_score = evaluate_file_precision('HarryPotter-en_to_fr.txt', 'HarryPotter-fr.txt')
fr_to_en_score = evaluate_file_precision('HarryPotter-fr_to_en.txt', 'HarryPotter-en.txt')

print(f"Average Unigram Precision (EN -> FR): {en_to_fr_score:.4f}")
print(f"Average Unigram Precision (FR -> EN): {fr_to_en_score:.4f}")

Average Unigram Precision (EN -> FR): 0.3940
Average Unigram Precision (FR -> EN): 0.4331


- English is generally a "simplier" target language so the larger precision makes sense here. There are less options per input. 
- French as a lot more verb as well as language components that don't exists in english, i.e. gender. 
- Likely a data bias in the underlying models. These models are trained on massive corpora of which there is an extreme adbundance on English to learn from. 

---

### 3. One easy way to cheat on simple precision scores by just providing very common words in a translation. What is the most frequently occurring English word in the sample? What is the most frequently occurring French word in the sample? Compute average unigram precision for some nonsense translations made up of very common English/French words.

In [2]:
from collections import Counter

def get_most_frequent_word(filename):
    """Identifies the most common unigram in a text file."""
    with open(filename, 'r', encoding='utf-8') as f:
        words = f.read().lower().split()
        return Counter(words).most_common(1)[0][0]

def compute_nonsense_precision(ref_file, common_word):
    """Calculates precision for a nonsense translation of 'common_word' repeated."""
    with open(ref_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    
    total_precision = 0
    for line in lines:
        ref_line = line.strip()
        ref_words = ref_line.split()
        if not ref_words: continue
        
        # Create nonsense: 'the the the...' matching reference length
        nonsense_cand = " ".join([common_word] * len(ref_words))
        total_precision += compute_unigram_precision(nonsense_cand, ref_line)
    
    return total_precision / len(lines)

# 1. Find the most frequent words
most_freq_en = get_most_frequent_word('HarryPotter-en.txt')
most_freq_fr = get_most_frequent_word('HarryPotter-fr.txt')

print(f"Most frequent English word: '{most_freq_en}'")
print(f"Most frequent French word: '{most_freq_fr}'")

# 2. Run the 'Cheating' evaluation
en_cheat_score = compute_nonsense_precision('HarryPotter-en.txt', most_freq_en)
fr_cheat_score = compute_nonsense_precision('HarryPotter-fr.txt', most_freq_fr)

print(f"Nonsense Precision (EN): {en_cheat_score:.4f}")
print(f"Nonsense Precision (FR): {fr_cheat_score:.4f}")

Most frequent English word: 'and'
Most frequent French word: 'et'
Nonsense Precision (EN): 0.4444
Nonsense Precision (FR): 0.4000


---

### 4. Modified unigram precision (Papineni et al., 2002) is not fooled by very short translations or translations which simply repeat frequent words. Implement this measure and run it on both the automatic translations provided by Google Translate and your nonsense translations.

This metric is the foundational component of the **BLEU** score.

The standard precision was fooled because it only checked if a word existed in the reference. Modified precision adds a **clipping** mechanism:
1. Count how many times each word appears in the **candidate** translation.
2. Count how many times each word appears in the **reference** translation.
3. For each word, "clip" the candidate count so it cannot exceed the reference count.
4. Sum these clipped counts and divide by the total number of words in the candidate.

In [3]:
from collections import Counter

def compute_modified_unigram_precision(candidate, reference):
    """
    Computes modified precision to prevent 'cheating' by repetition.
    """
    cand_words = candidate.lower().split()
    ref_words = reference.lower().split()
    
    if not cand_words:
        return 0
    
    # Step 1 & 2: Get frequency counts for both
    cand_counts = Counter(cand_words)
    ref_counts = Counter(ref_words)
    
    clipped_matches = 0
    
    # Step 3: For each word in the machine translation, 
    # clip its count by the max frequency in the reference
    for word, count in cand_counts.items():
        max_ref_count = ref_counts.get(word, 0)
        clipped_matches += min(count, max_ref_count)
    
    # Step 4: Divide by total unigrams in candidate
    return clipped_matches / len(cand_words)

def evaluate_file_modified(candidate_file, reference_file):
    """Averages modified unigram precision across a file."""
    with open(candidate_file, 'r', encoding='utf-8') as cand_f, \
         open(reference_file, 'r', encoding='utf-8') as ref_f:
        
        cand_lines = cand_f.readlines()
        ref_lines = ref_f.readlines()
        
        total_precision = 0
        for cand, ref in zip(cand_lines, ref_lines):
            total_precision += compute_modified_unigram_precision(cand.strip(), ref.strip())
            
        return total_precision / len(cand_lines)

# --- TESTING ---

# 1. Automatic Translations
mod_en_to_fr = evaluate_file_modified('HarryPotter-en_to_fr.txt', 'HarryPotter-fr.txt')
mod_fr_to_en = evaluate_file_modified('HarryPotter-fr_to_en.txt', 'HarryPotter-en.txt')

print(f"Modified Precision (EN -> FR): {mod_en_to_fr:.4f}")
print(f"Modified Precision (FR -> EN): {mod_fr_to_en:.4f}")

# 2. Nonsense Translations (The 'Cheating' test)
# Replace 'the' and 'de' with the most frequent words you found in Task 3
nonsense_en_score = 0
with open('HarryPotter-en.txt', 'r') as f:
    for line in f:
        # Create a line of just 'the'
        length = len(line.split())
        nonsense_line = " ".join([most_freq_en] * length)
        nonsense_en_score += compute_modified_unigram_precision(nonsense_line, line)
print(f"Nonsense Modified Precision (EN): {nonsense_en_score/10:.4f}")

Modified Precision (EN -> FR): 0.3792
Modified Precision (FR -> EN): 0.4103
Nonsense Modified Precision (EN): 0.0204


- The results for the translations stay largely the same as we didn't have an intristic bias towards duplicating common words
- But the nonsense sequence as a precision which identifies that it is a useless translation.

---

### 5. Unigram precision (modified or otherwise) does not take into account the ordering of the translated words. Extend your code to also compute the average bigram precision and average trigram precision. Combine these scores by averaging. Do you think it is necessary to compute modified precision in the case of bigrams and trigrams?

Shift the focus from individual word matches to phrasal matches, which measures the fluency and word order of the translation.

This means we need to create a method to collection n-grams on a sliding window basis. 

The logic remains the same as Task 4, but instead of matching single words, match the generated n-gram strings. 

- Count how many bigrams appear in the candidate translation.
- Match them against the bigrams present in the gold standard reference.
- Calculate Precision: $\frac{\text{Matching N-grams}}{\text{Total N-grams in Candidate}}$.

In term of averaging the results a standard evaluation like BLEU, researchers typically use the geometric mean of the precisions ($p_1$ to $p_4$) rather than a simple arithmetic mean, as it more heavily penalizes a score of zero in any single category. For this lab, a simple arithmetic average of your unigram, bigram, and trigram scores is sufficient. Just to be clear, the averaging comes looking at different levels of n-gram and averaging the performance.

For n-grams is becomes less of an issue to **clip**. This is because n-grams because increasingly sparse in real text. There is unlikely to be n-grams in the text that can be effectly gamed for a high performance. That being said, there is no harm in clipping. Metrics like BLEU still clip. 

> When implementing your n-gram function, remember that a sentence with $W$ words will have $W - 1$ bigrams and $W - 2$ trigrams. Ensure your precision denominator accounts for this correctly to avoid division-by-zero errors in very short sentences.

In [4]:
from collections import Counter

def get_ngrams(text, n):
    """
    Generates a list of n-grams from a string.
    Example: 'the cat sat', n=2 -> ['the cat', 'cat sat']
    """
    words = text.lower().split()
    # The zip function creates a sliding window of n words
    ngrams = [ " ".join(words[i:i+n]) for i in range(len(words)-n+1) ]
    return ngrams

def compute_modified_ngram_precision(candidate, reference, n):
    """
    Computes modified precision for any n-gram order.
    """
    cand_ngrams = get_ngrams(candidate, n)
    ref_ngrams = get_ngrams(reference, n)
    
    if not cand_ngrams:
        return 0
    
    cand_counts = Counter(cand_ngrams)
    ref_counts = Counter(ref_ngrams)
    
    clipped_matches = 0
    for ngram, count in cand_counts.items():
        max_ref_count = ref_counts.get(ngram, 0)
        clipped_matches += min(count, max_ref_count)
    
    return clipped_matches / len(cand_ngrams)

def evaluate_combined_score(candidate_file, reference_file):
    """
    Computes average unigram, bigram, and trigram precision 
    and returns their mean.
    """
    with open(candidate_file, 'r', encoding='utf-8') as cand_f, \
         open(reference_file, 'r', encoding='utf-8') as ref_f:
        
        cand_lines = cand_f.readlines()
        ref_lines = ref_f.readlines()
        
        totals = {1: 0, 2: 0, 3: 0} # Accumulators for n=1, 2, 3
        
        for cand, ref in zip(cand_lines, ref_lines):
            cand, ref = cand.strip(), ref.strip()
            for n in [1, 2, 3]:
                totals[n] += compute_modified_ngram_precision(cand, ref, n)
        
        # Calculate averages for each N
        avg_scores = {n: totals[n] / len(cand_lines) for n in [1, 2, 3]}
        
        # Final combined score (Arithmetic mean)
        combined = sum(avg_scores.values()) / len(avg_scores)
        
        return avg_scores, combined

# Run the evaluation
en_fr_avgs, en_fr_combined = evaluate_combined_score('HarryPotter-en_to_fr.txt', 'HarryPotter-fr.txt')
fr_en_avgs, fr_en_combined = evaluate_combined_score('HarryPotter-fr_to_en.txt', 'HarryPotter-en.txt')

print(f"EN->FR Scores: Unigram: {en_fr_avgs[1]:.4f}, Bigram: {en_fr_avgs[2]:.4f}, Trigram: {en_fr_avgs[3]:.4f}")
print(f"EN->FR Combined Score: {en_fr_combined:.4f}")

print(f"\nFR->EN Scores: Unigram: {fr_en_avgs[1]:.4f}, Bigram: {fr_en_avgs[2]:.4f}, Trigram: {fr_en_avgs[3]:.4f}")
print(f"FR->EN Combined Score: {fr_en_combined:.4f}")

EN->FR Scores: Unigram: 0.3792, Bigram: 0.1997, Trigram: 0.1287
EN->FR Combined Score: 0.2358

FR->EN Scores: Unigram: 0.4103, Bigram: 0.2316, Trigram: 0.1833
FR->EN Combined Score: 0.2751


**The Decay Pattern:** You will notice that as $n$ increases, the score drops significantly.

**Reason:** It is much harder for a machine to get three words in the exact right order than it is to get one word right.

**Fluency vs. Accuracy:** The unigram score measures if the model knows the right vocabulary (Accuracy), while the bigram/trigram scores measure if the model knows how to construct a sentence (Fluency).

## 3 Extensions

1. Can you modify / extend your code to implement the BLEU score as described by Papineni et al. (2002)?

2. An alternative way of evaluating automatic translations is by considering how much post-editing is required to correct the automatic translation into something which is both faithful and fluent. One way of assessing post-editing effort is by computing the HTER score (see http://languagelog.ldc.upenn.edu/nll/?p=193) - count the number of substitutions, insertions, deletions and shifts required and divide by the number of words in the gold standard translation. Can you compute the HTER scores for the French-English translation? Do you think you could automate this?

## References

Kishore Papineni, Salim Roukos, Todd ward, and Wei-Jing Zhu. 2002. Bleu: A method for automatic evaluation of machine translation. In Proceedings of ACL.
